In [1]:

# ============================================
# تنظيف بيانات جمعية أعمال لعامي 2024 و2025
# باستخدام مكتبة Pandas في Google Colab
# ============================================

import pandas as pd
import numpy as np
import re
from google.colab import files

# --------------------------------------------
# 1. رفع ملف CSV
# --------------------------------------------

print("ارفعي ملف CSV الخاص بجمعية أعمال")
uploaded = files.upload()

file_name = next(iter(uploaded))

# قراءة الملف مع الحفاظ على القيم الأصلية قدر الإمكان
df = pd.read_csv(
    file_name,
    dtype=str,
    encoding="utf-8-sig"
)

# تنظيف أسماء الأعمدة من المسافات الزائدة
df.columns = df.columns.str.strip()

# التأكد من وجود الأعمدة المطلوبة
required_columns = ["المجال", "2024", "2025"]

missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"الأعمدة التالية غير موجودة في الملف: {missing_columns}"
    )

# الاحتفاظ بالأعمدة المطلوبة وبترتيبها
df = df[required_columns].copy()

print("\n========== فحص البيانات قبل التنظيف ==========")
print("عدد الصفوف:", len(df))
print("عدد الأعمدة:", len(df.columns))

print("\nالقيم المفقودة في كل عمود:")
print(df.isna().sum())

print("\nأول خمسة صفوف:")
print(df.head())


# --------------------------------------------
# 2. حذف السجلات الفارغة بالكامل
# --------------------------------------------

before_empty = len(df)

# اعتبار النصوص التي تحتوي على مسافات فقط فارغة
df = df.replace(r"^\s*$", np.nan, regex=True)

df = df.dropna(how="all").copy()

after_empty = len(df)

print("\n========== حذف السجلات الفارغة ==========")
print("عدد الصفوف قبل الحذف:", before_empty)
print("عدد السجلات الفارغة المحذوفة:", before_empty - after_empty)
print("عدد الصفوف بعد الحذف:", after_empty)


# --------------------------------------------
# 3. تنظيف المسافات الزائدة في النصوص
# --------------------------------------------

for col in df.columns:
    df[col] = df[col].map(
        lambda value: value.strip()
        if isinstance(value, str)
        else value
    )

# تحويل النصوص الفارغة إلى قيم مفقودة
df = df.replace(r"^\s*$", np.nan, regex=True)

print("\nتم تنظيف المسافات الزائدة.")


# --------------------------------------------
# 4. فحص التكرارات الكاملة وحذفها
# --------------------------------------------

duplicate_count = df.duplicated().sum()

print("\n========== فحص التكرارات ==========")
print("عدد السجلات المكررة بالكامل:", duplicate_count)

if duplicate_count > 0:
    print("\nالسجلات المكررة قبل الحذف:")
    print(df[df.duplicated(keep=False)])

    df = df.drop_duplicates().copy()

    print("\nتم حذف التكرارات الكاملة فقط.")
else:
    print("لا توجد تكرارات كاملة.")

print("عدد الصفوف بعد فحص التكرارات:", len(df))


# --------------------------------------------
# 5. توحيد القيم الرقمية وتجهيزها للتحليل
# --------------------------------------------

def clean_numeric(value):
    """
    تنظيف القيم الرقمية مع الحفاظ على معنى النسب المئوية.

    أمثلة:
    1,200 ريال  -> 1200
    35 %        -> 35
    35%         -> 35
    1,200.50    -> 1200.50
    """

    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text == "":
        return np.nan

    # توحيد بعض الأرقام العربية والفارسية إلى أرقام إنجليزية
    translation_table = str.maketrans(
        "٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹٫٬",
        "01234567890123456789.,"
    )

    text = text.translate(translation_table)

    # إزالة الفواصل المستخدمة للفصل بين الآلاف
    text = text.replace(",", "")

    # استخراج الجزء الرقمي، مع السماح بالإشارة العشرية والسالبة
    # وأقواس الأرقام السالبة مثل (1200)
    negative_parentheses = (
        text.startswith("(") and text.endswith(")")
    )

    if negative_parentheses:
        text = text[1:-1].strip()

    # البحث عن رقم في النص وتجاهل الكلمات والوحدات المحيطة به
    match = re.search(
        r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?",
        text
    )

    if not match:
        return np.nan

    number_text = match.group()

    try:
        number = float(number_text)

        if negative_parentheses:
            number = -abs(number)

        return number

    except ValueError:
        return np.nan


# الاحتفاظ بنسخة من القيم قبل تحويلها
original_numeric_values = df[["2024", "2025"]].copy()

for col in ["2024", "2025"]:
    df[col] = df[col].apply(clean_numeric)

print("\n========== تحويل الأعمدة الرقمية ==========")
print("تم تحويل عمودي 2024 و2025 إلى قيم رقمية.")
print("عمود المجال سيبقى نصيًا.")

# عرض القيم التي لم يمكن تحويلها، قبل اتخاذ أي قرار بشأنها
print("\nالقيم الأصلية التي لم يمكن تحويلها إلى أرقام:")

for col in ["2024", "2025"]:
    original = original_numeric_values[col].fillna("").astype(str).str.strip()

    invalid_mask = (
        original.ne("")
        & df[col].isna()
    )

    if invalid_mask.any():
        print(f"\nالعمود {col}:")
        print(
            pd.DataFrame({
                "المجال": df.loc[invalid_mask, "المجال"],
                "القيمة الأصلية": original.loc[invalid_mask]
            })
        )
    else:
        print(f"{col}: لا توجد قيم غير قابلة للتحويل.")


# --------------------------------------------
# 6. عرض السجلات التي تحتوي على 444444
# --------------------------------------------

# البحث في القيم الأصلية قبل تحويلها إلى أرقام
error_mask = original_numeric_values.apply(
    lambda col: col.str.contains(
        r"(?<!\d)444444(?!\d)",
        na=False,
        regex=True
    )
).any(axis=1)

print("\n========== فحص القيمة الخاطئة 444444 ==========")

if error_mask.any():
    print("السجلات التي تحتوي على 444444 قبل الحذف:")
    print(
        pd.concat(
            [
                df.loc[error_mask],
                original_numeric_values.loc[error_mask].add_suffix("_قبل_التنظيف")
            ],
            axis=1
        )
    )

    # حذف السجلات المؤكدة الخطأ فقط
    df = df.loc[~error_mask].copy()

    print("\nتم حذف السجلات التي تحتوي على 444444.")
    print("عدد السجلات المحذوفة:", int(error_mask.sum()))

else:
    print("لم يتم العثور على القيمة 444444.")

print("عدد الصفوف الحالي:", len(df))


# --------------------------------------------
# 7. عرض السجلات المشكوك فيها للمراجعة
# --------------------------------------------

# لا نحذف القيم الغريبة تلقائيًا.
# نعرض السجلات التي تحتوي على قيم مفقودة أو سالبة
# أو أرقام كبيرة جدًا للمراجعة فقط.

review_mask = (
    df[["2024", "2025"]].isna().any(axis=1)
    | (df[["2024", "2025"]] < 0).any(axis=1)
    | (df[["2024", "2025"]] > 1_000_000_000).any(axis=1)
    | df["المجال"].isna()
)

print("\n========== السجلات التي تحتاج إلى مراجعة ==========")

if review_mask.any():
    print(
        df.loc[review_mask].to_string(index=True)
    )
    print(
        "\nهذه السجلات لم تُحذف تلقائيًا. "
        "راجعيها وحددي ما إذا كانت صحيحة."
    )
else:
    print("لم تظهر سجلات وفق قواعد الفحص الأولية.")

# ملاحظة: حد المليار مجرد تنبيه أولي، وليس حكمًا بأن القيمة خاطئة.


# --------------------------------------------
# 8. فحص القيم المفقودة بعد التنظيف
# --------------------------------------------

print("\n========== القيم المفقودة بعد التنظيف ==========")
print(df.isna().sum())


# --------------------------------------------
# 9. عرض أنواع البيانات وحجم الجدول النهائي
# --------------------------------------------

# التأكد من أن المجال نصي، والسنتين رقميتان
df["المجال"] = df["المجال"].astype("string")

df["2024"] = pd.to_numeric(df["2024"], errors="coerce")
df["2025"] = pd.to_numeric(df["2025"], errors="coerce")

print("\n========== أنواع البيانات النهائية ==========")
print(df.dtypes)

print("\nعدد الصفوف النهائي:", len(df))
print("عدد الأعمدة النهائي:", len(df.columns))

print("\nالجدول النهائي بعد التنظيف:")
print(df.to_string(index=False))


# --------------------------------------------
# 10. حفظ الملف الجديد وتنزيله
# --------------------------------------------

output_file = "data_cleaning_AI.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nتم حفظ الملف النهائي باسم:", output_file)

# تنزيل الملف مباشرة إلى جهازك
files.download(output_file)


ارفعي ملف CSV الخاص بجمعية أعمال


Saving Data2024-2025.csv to Data2024-2025.csv

========== فحص البيانات قبل التنظيف ==========
عدد الصفوف: 17
عدد الأعمدة: 3

القيم المفقودة في كل عمود:
المجال    3
2024      3
2025      3
dtype: int64

أول خمسة صفوف:
                          المجال        2024          2025
0                 عدد المستفيدين        8885          7964
1                     فرص التطوع         100      365 فرصة
2               عدد ساعات التطوع        3426          4900
3                            NaN         NaN           NaN
4  المستفيدين من التمويل الجمعية  850 مستفيد  1119 مستفيد 

========== حذف السجلات الفارغة ==========
عدد الصفوف قبل الحذف: 17
عدد السجلات الفارغة المحذوفة: 3
عدد الصفوف بعد الحذف: 14

تم تنظيف المسافات الزائدة.

========== فحص التكرارات ==========
عدد السجلات المكررة بالكامل: 0
لا توجد تكرارات كاملة.
عدد الصفوف بعد فحص التكرارات: 14

========== تحويل الأعمدة الرقمية ==========
تم تحويل عمودي 2024 و2025 إلى قيم رقمية.
عمود المجال سيبقى نصيًا.

القيم الأصلية التي لم يمكن تحويلها إلى أ

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>